In [18]:
import pandas as pd
import numpy as np

In [19]:
# Load the data
df = pd.read_csv("workplace_data_cleaning_practice.csv")

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates(subset=["employee_id"])

In [21]:
# Remove leading and trailing whitespace
text_columns = ["employee_id", "department", "office_location"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [22]:
# Standardize department names
df["department"] = df["department"].str.casefold()

In [23]:
# Standardize office locations
df["office_location"] = df["office_location"].str.casefold()

In [24]:
# Convert numeric columns to numeric data types
numeric_columns = [
    "job_satisfaction_score",
    "weekly_hours",
    "projects_completed",
    "annual_training_hours",
    "commute_minutes",
    "annual_salary_usd"
]

In [25]:
# Standardize common missing-value markers
df = df.replace(
    ["N/A", "n/a", "unknown", "Unknown", ""],
    np.nan
)

In [26]:
# Remove currency formatting
df["annual_salary_usd"] = df["annual_salary_usd"].astype("string").str.replace('[^a-zA-Z0-9]', '', regex=False)
    

In [27]:
# Correct invalid satisfaction scores
# Scores are supposed to range from 1 to 10.
invalid_scores = ~df["job_satisfaction_score"].between(1, 10)
df.loc[invalid_scores, "job_satisfaction_score"] = np.nan

In [28]:
# Flag negative values in nonnegative columns
nonnegative_columns = [
    "weekly_hours",
    "projects_completed",
    "annual_training_hours",
    "commute_minutes",
    "annual_salary_usd"
]

for col in nonnegative_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[df[col] < 0, col] = np.nan

In [29]:
# Remove outliers
columns = [
    "weekly_hours",
    "commute_minutes",
    "annual_training_hours"
]

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[df[col].between(lower, upper) | df[col].isna()]

In [30]:
# Impute missing values
for col in numeric_columns:
    x = df[col].mean()
    df.fillna({col: x}, inplace=True)

In [31]:
# 11. Save the cleaned dataset
df.to_csv("workplace_data_cleaned.csv", index=False)